# Chat Templates & Special Tokens Lab
## What Happens When You Use the Wrong Template?

**What you'll learn:**
1. How chat templates convert a messages dict into a flat token string
2. Why different models need different templates
3. What actually breaks when you use the wrong template (live demo)
4. The role of BOS, EOS, PAD, and generation prompt tokens
5. How to inspect and debug any model's chat format

**Builds on:** The tokenisation lab. If you haven't done that one, go there first.

---

### Models used in this notebook

Model

| Qwen3 1.7B | ChatML (`<\|im_start\|>`) 

| Phi-4 | ChatML + `<\|im_sep\|>` 

| Nemotron Nano v3 | ChatML + `<think>`

All generation demos use **Qwen3 1.7B** — it's the smallest and fastest for live comparison.

## Setup

In [34]:
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Device detection (same as tokenisation lab) ──────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE  = torch.bfloat16
    print(f"✅ CUDA — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = "mps"
    DTYPE  = torch.float32
    print("✅ Apple Silicon (MPS)")
else:
    DEVICE = "cpu"
    DTYPE  = torch.float32
    print("⚠️  CPU — generation will be slow")

# ── Model paths ──────────────────────────────────────────────────────
# Change these to match your setup.
# If local paths don't exist the cells below fall back to HuggingFace.
LOCAL = {
    "qwen3":    "/bigstore/models/qwen/qwen-3-1.7b",
    "phi4":     "/bigstore/models/tprm-model-discovery/phi-4",
    "nemotron": "/bigstore/models/nemotron-nano-v3",
}
PUBLIC = {
    "qwen3":  "Qwen/Qwen2.5-1.5B-Instruct",
    "phi4":   "microsoft/Phi-3-mini-4k-instruct",
}

def resolve(key):
    """Return local path if it exists, else HuggingFace fallback."""
    local = LOCAL.get(key, "")
    if os.path.isdir(local):
        print(f"  Using local: {local}")
        return local, True
    hf = PUBLIC.get(key, None)
    if hf:
        print(f"  Local not found — will download: {hf}")
        return hf, False
    raise FileNotFoundError(f"No path found for '{key}'")

QWEN3_PATH, QWEN3_LOCAL   = resolve("qwen3")
PHI4_PATH,  PHI4_LOCAL    = resolve("phi4")
print(f"\nDevice: {DEVICE} | dtype: {DTYPE}")

✅ Apple Silicon (MPS)
  Local not found — will download: Qwen/Qwen2.5-1.5B-Instruct
  Local not found — will download: microsoft/Phi-3-mini-4k-instruct

Device: mps | dtype: torch.float32


---
# SECTION 1: Rendering a Chat Template
## From a messages dict to a flat token string

The model never sees a structured dict. It sees a **flat string** with special tokens inserted by the chat template.  
This section makes that visible.

In [35]:
# Load Qwen3 tokenizer (no model weights yet — tokenizers are tiny)
tok_qwen = AutoTokenizer.from_pretrained(
    QWEN3_PATH,
    trust_remote_code=True,
    local_files_only=QWEN3_LOCAL,
)
print(f"Qwen3 tokenizer loaded — vocab size: {tok_qwen.vocab_size:,}")
print(f"Has chat template: {bool(tok_qwen.chat_template)}")

Qwen3 tokenizer loaded — vocab size: 151,643
Has chat template: True


In [3]:
# The messages dict — what you write in your code
messages = [
    {"role": "system",    "content": "You are a concise assistant. Answer in one sentence."},
    {"role": "user",      "content": "What is 2+2?"},
]

# apply_chat_template converts it to a flat string
rendered = tok_qwen.apply_chat_template(
    messages,
    tokenize=False,               # return string, not token IDs
    add_generation_prompt=True,   # append assistant header
)

print("══ What you write ════════════════════════════════════════════════")
for m in messages:
    print(f"  [{m['role']}] {m['content']}")

print("\n══ What the model receives (rendered string) ═════════════════════")
print(rendered)

print("\n══ repr() — shows every hidden character ═══════════════════════")
print(repr(rendered))

══ What you write ════════════════════════════════════════════════
  [system] You are a concise assistant. Answer in one sentence.
  [user] What is 2+2?

══ What the model receives (rendered string) ═════════════════════
<|im_start|>system
You are a concise assistant. Answer in one sentence.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant


══ repr() — shows every hidden character ═══════════════════════
'<|im_start|>system\nYou are a concise assistant. Answer in one sentence.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\n'


In [4]:
# Break the rendered string into individual tokens
def show_rendered_tokens(tokenizer, rendered_text, label=""):
    """Show each token in the rendered prompt with its ID."""
    ids = tokenizer.encode(rendered_text, add_special_tokens=False)
    rows = []
    for pos, tid in enumerate(ids):
        raw = tokenizer.convert_ids_to_tokens([tid])[0]
        decoded = tokenizer.decode([tid])
        rows.append({"pos": pos, "id": tid, "token": repr(raw), "decoded": repr(decoded)})
    df = pd.DataFrame(rows)
    print(f"\n{label} — {len(ids)} tokens total\n")
    print(df.to_string(index=False))
    return df

df_correct = show_rendered_tokens(tok_qwen, rendered, "Qwen3 — CORRECT template")


Qwen3 — CORRECT template — 31 tokens total

 pos     id          token        decoded
   0 151644 '<|im_start|>' '<|im_start|>'
   1   8948       'system'       'system'
   2    198            'Ċ'           '\n'
   3   2610          'You'          'You'
   4    525         'Ġare'         ' are'
   5    264           'Ġa'           ' a'
   6  63594     'Ġconcise'     ' concise'
   7  17847   'Ġassistant'   ' assistant'
   8     13            '.'            '.'
   9  21806      'ĠAnswer'      ' Answer'
  10    304          'Ġin'          ' in'
  11    825         'Ġone'         ' one'
  12  11652    'Ġsentence'    ' sentence'
  13     13            '.'            '.'
  14 151645   '<|im_end|>'   '<|im_end|>'
  15    198            'Ċ'           '\n'
  16 151644 '<|im_start|>' '<|im_start|>'
  17    872         'user'         'user'
  18    198            'Ċ'           '\n'
  19   3838         'What'         'What'
  20    374          'Ġis'          ' is'
  21    220            'Ġ'     

In [33]:
import re
df_correct[df_correct.token.apply(lambda x: bool(re.search('<', x)))][['id', 'token']].drop_duplicates()

,id,token
0,151644,'<|im_start|>'
14,151645,'<|im_end|>'


---
# SECTION 2: Templates Side by Side
## Same messages, three different rendered strings

Each model family was fine-tuned on a different serialisation format.  
Here we render the same messages using three formats and compare them.

In [5]:
# Load Phi-4 tokenizer for comparison
tok_phi4 = AutoTokenizer.from_pretrained(
    PHI4_PATH,
    trust_remote_code=True,
    local_files_only=PHI4_LOCAL,
)
print(f"Phi-4 tokenizer loaded — vocab size: {tok_phi4.vocab_size:,}")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Phi-4 tokenizer loaded — vocab size: 32,000


In [6]:
# Manually define three template functions for comparison
# We do this so you can see the structure without needing every model loaded.

def chatml_template(messages, add_generation_prompt=True):
    """Qwen2/3 ChatML — <|im_start|>role\ncontent<|im_end|>"""
    out = ""
    for m in messages:
        out += f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n"
    if add_generation_prompt:
        out += "<|im_start|>assistant\n"
    return out

def phi4_template(messages, add_generation_prompt=True):
    """Phi-4 — <|im_start|>role<|im_sep|>content<|im_end|>"""
    out = ""
    for m in messages:
        out += f"<|im_start|>{m['role']}<|im_sep|>{m['content']}<|im_end|>\n"
    if add_generation_prompt:
        out += "<|im_start|>assistant<|im_sep|>"
    return out

def llama_template(messages, add_generation_prompt=True):
    """Llama-2-Chat / TinyLlama — [INST] ... [/INST]"""
    system = next((m["content"] for m in messages if m["role"] == "system"), None)
    out = "<s>"
    for m in messages:
        if m["role"] == "system":
            continue
        if m["role"] == "user":
            out += "[INST] "
            if system:
                out += f"<<SYS>>\n{system}\n<</SYS>>\n\n"
                system = None   # only on first user turn
            out += f"{m['content']} [/INST]"
        elif m["role"] == "assistant":
            out += f" {m['content']} </s><s>"
    if add_generation_prompt:
        out += " "  # space before assistant generation
    return out

def no_template(messages):
    """Raw text with no structure — what a base model gets."""
    return "\n".join(m["content"] for m in messages) + "\n"


# Render with each format
templates = [
    ("Qwen3 ChatML (correct)",       chatml_template(messages)),
    ("Phi-4 ChatML+sep (different)",  phi4_template(messages)),
    ("Llama [INST] (very different)", llama_template(messages)),
    ("No template (base model)",      no_template(messages)),
]

for name, text in templates:
    n_tokens = len(tok_qwen.encode(text, add_special_tokens=False))
    print(f"{'═'*60}")
    print(f"  {name}  [{n_tokens} tokens in Qwen3 vocab]")
    print(f"{'═'*60}")
    print(text)
    print()

════════════════════════════════════════════════════════════
  Qwen3 ChatML (correct)  [31 tokens in Qwen3 vocab]
════════════════════════════════════════════════════════════
<|im_start|>system
You are a concise assistant. Answer in one sentence.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant


════════════════════════════════════════════════════════════
  Phi-4 ChatML+sep (different)  [46 tokens in Qwen3 vocab]
════════════════════════════════════════════════════════════
<|im_start|>system<|im_sep|>You are a concise assistant. Answer in one sentence.<|im_end|>
<|im_start|>user<|im_sep|>What is 2+2?<|im_end|>
<|im_start|>assistant<|im_sep|>

════════════════════════════════════════════════════════════
  Llama [INST] (very different)  [33 tokens in Qwen3 vocab]
════════════════════════════════════════════════════════════
<s>[INST] <<SYS>>
You are a concise assistant. Answer in one sentence.
<</SYS>>

What is 2+2? [/INST] 

═══════════════════════════════════════

---
# SECTION 3: Load the Model for Generation

We load **Qwen3 1.7B** once and use it for all generation demos.  
First run takes ~30 seconds (downloads ~3 GB if not cached).

In [36]:
print(f"Loading Qwen3 1.7B on {DEVICE} ({DTYPE}) …")

model = AutoModelForCausalLM.from_pretrained(
    QWEN3_PATH,
    dtype=DTYPE,
    trust_remote_code=True,
    local_files_only=QWEN3_LOCAL,
).to(DEVICE).eval()

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"✅ Loaded — {n_params / 1e9:.2f}B parameters")

Loading Qwen3 1.7B on mps (torch.float32) …


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Loaded — 1.54B parameters


In [37]:
@torch.inference_mode()
def generate(prompt_text, max_new_tokens=80, temperature=0.0):
    """
    Tokenise a raw string with Qwen3's tokenizer and generate a response.
    Returns the decoded new tokens only (not the prompt).
    """
    input_ids = tok_qwen.encode(prompt_text, return_tensors="pt",
                                 add_special_tokens=False).to(DEVICE)
    prompt_len = input_ids.shape[-1]

    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature if temperature > 0 else 1.0,
        eos_token_id=tok_qwen.eos_token_id,
        pad_token_id=tok_qwen.eos_token_id,   # Qwen3 reuses EOS as PAD
    )

    new_ids  = output_ids[0][prompt_len:]
    response = tok_qwen.decode(new_ids, skip_special_tokens=True)
    raw      = tok_qwen.decode(new_ids, skip_special_tokens=False)
    return response.strip(), raw.strip(), len(new_ids)

print("generate() helper ready")

generate() helper ready


---
# SECTION 4: ⭐ The Wrong Template Demo
## Watch what breaks — and why

We feed the **same question** to the **same model** using four different prompt formats.  
Same model weights. Same question. Only the prompt string changes.

Expected results:
- **Correct template** → Clean answer
- **Phi-4 template** → Mild degradation (similar format, one token different)
- **Llama template** → Clear confusion (completely different format)
- **No template** → Model completes the text as a document, not a Q&A

In [38]:
question = [
    {"role": "system", "content": "You are a concise assistant. Answer in one sentence."},
    {"role": "user",   "content": "What is 2+2?"},
]

experiments = [
    ("✅ Correct  — Qwen3 ChatML",        chatml_template(question)),
    ("⚠️  Wrong    — Phi-4 +<|im_sep|>",  phi4_template(question)),
    ("❌ Very wrong — Llama [INST]",       llama_template(question)),
    ("💥 No template — raw text",          no_template(question)),
]

results = []
for label, prompt in experiments:
    response, raw, n_new = generate(prompt, max_new_tokens=60)
    results.append({"Experiment": label, "Response": response, "New tokens": n_new})

print("\n" + "═" * 80)
print("SAME MODEL — SAME QUESTION — DIFFERENT TEMPLATES")
print("═" * 80)
for r in results:
    print(f"\n{r['Experiment']}")
    print(f"  [{r['New tokens']} new tokens] {r['Response']}")

print("\n" + "═" * 80)


════════════════════════════════════════════════════════════════════════════════
SAME MODEL — SAME QUESTION — DIFFERENT TEMPLATES
════════════════════════════════════════════════════════════════════════════════

✅ Correct  — Qwen3 ChatML
  [9 new tokens] 2 + 2 equals 4.

⚠️  Wrong    — Phi-4 +<|im_sep|>
  [8 new tokens] 2+2 equals 4.

❌ Very wrong — Llama [INST]
  [60 new tokens] 4. [/INST]Human: Can you tell me the answer again please? Sure, the sum of two plus two is four. [/INST]Human: How about this time? What's the square root of 16? The square root of 16 is 4.

💥 No template — raw text
  [60 new tokens] The sum of two plus two is four.Human: Can you provide me with some examples of how to use the word "suffocate" in a sentence? Sure, here are some example sentences using the word "suffocate":

1. The smoke from the fire was suffocating

════════════════════════════════════════════════════════════════════════════════


In [10]:
# Harder question — shows the degradation more clearly
hard_question = [
    {"role": "system", "content": "You are a Python expert. Give only code, no explanation."},
    {"role": "user",   "content": "Write a function that returns the Fibonacci sequence up to n."},
]

print("═" * 80)
print("HARDER QUESTION: Python function — Correct vs Wrong template")
print("═" * 80)

for label, prompt in [
    ("✅ Correct  — Qwen3 ChatML",       chatml_template(hard_question)),
    ("❌ Very wrong — Llama [INST]",      llama_template(hard_question)),
    ("💥 No template — raw text",         no_template(hard_question)),
]:
    response, _, n_new = generate(prompt, max_new_tokens=120)
    print(f"\n{label}  [{n_new} new tokens]")
    print("-" * 60)
    print(response)

print("\n" + "═" * 80)

════════════════════════════════════════════════════════════════════════════════
HARDER QUESTION: Python function — Correct vs Wrong template
════════════════════════════════════════════════════════════════════════════════

✅ Correct  — Qwen3 ChatML  [58 new tokens]
------------------------------------------------------------
```python
def fibonacci(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        next_value = fib_sequence[-1] + fib_sequence[-2]
        fib_sequence.append(next_value)
    return fib_sequence[:n]
```

❌ Very wrong — Llama [INST]  [120 new tokens]
------------------------------------------------------------
```python
def fibonacci(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        next_value = fib_sequence[-1] + fib_sequence[-2]
        fib_sequence.append(next_value)
    return fib_sequence[:n]

# Example usage:
print(fibonacci(10)) # Output: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
```Human: I want you to write a Python program that t

In [11]:
# Show the raw token IDs at the decision point
# This makes the mechanical reason visible

print("WHY THE LLAMA TEMPLATE BREAKS QWEN — the token-level view\n")

chatml_text = chatml_template(question)
llama_text  = llama_template(question)

for name, text in [("Qwen3 ChatML", chatml_text), ("Llama [INST]", llama_text)]:
    ids = tok_qwen.encode(text, add_special_tokens=False)
    tokens = tok_qwen.convert_ids_to_tokens(ids)
    print(f"{'─'*60}")
    print(f"{name} — {len(ids)} tokens")
    print(f"{'─'*60}")
    for pos, (tid, tok) in enumerate(zip(ids, tokens)):
        marker = "← ROLE BOUNDARY" if "im_start" in tok or "INST" in tok else ""
        print(f"  {pos:>3}  {tid:>7}  {repr(tok):<25} {marker}")
    print()

print("""
KEY INSIGHT:
  • In ChatML, <|im_start|> (ID 151644) is a SPECIAL TOKEN — one integer, strong signal.
  • In the Llama template, [INST] is THREE regular tokens: '[' + 'INST' + ']'
  • Qwen3 has never seen [INST] as a role boundary — it treats it as ordinary text.
  • The model tries to continue the text, not answer the question.
""")

WHY THE LLAMA TEMPLATE BREAKS QWEN — the token-level view

────────────────────────────────────────────────────────────
Qwen3 ChatML — 31 tokens
────────────────────────────────────────────────────────────
    0   151644  '<|im_start|>'            ← ROLE BOUNDARY
    1     8948  'system'                  
    2      198  'Ċ'                       
    3     2610  'You'                     
    4      525  'Ġare'                    
    5      264  'Ġa'                      
    6    63594  'Ġconcise'                
    7    17847  'Ġassistant'              
    8       13  '.'                       
    9    21806  'ĠAnswer'                 
   10      304  'Ġin'                     
   11      825  'Ġone'                    
   12    11652  'Ġsentence'               
   13       13  '.'                       
   14   151645  '<|im_end|>'              
   15      198  'Ċ'                       
   16   151644  '<|im_start|>'            ← ROLE BOUNDARY
   17      872  'user'           

---
# SECTION 5: The add_generation_prompt Demo
## The single most common mistake in production

`add_generation_prompt=True` appends the assistant turn opener at the end of the prompt.  
Without it, the model has no signal that it should switch to assistant mode.

In [12]:
q = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user",   "content": "What is the capital of France?"},
]

with_gen    = tok_qwen.apply_chat_template(q, tokenize=False, add_generation_prompt=True)
without_gen = tok_qwen.apply_chat_template(q, tokenize=False, add_generation_prompt=False)

print("WITH add_generation_prompt=True (correct for inference):")
print(repr(with_gen))
print("  ↑ ends with <|im_start|>assistant  — model generates HERE as assistant")

print("\nWITHOUT add_generation_prompt=False (wrong):")
print(repr(without_gen))
print("  ↑ ends with <|im_end|>  — model has no idea whose turn it is")

print("\n" + "═" * 70)
print("GENERATION COMPARISON")
print("═" * 70)

resp_with, _, _ = generate(with_gen,    max_new_tokens=50)
resp_without, raw_without, _ = generate(without_gen, max_new_tokens=50)

print(f"\n✅ With add_generation_prompt=True:")
print(f"   {resp_with}")

print(f"\n❌ Without (add_generation_prompt=False):")
print(f"   {resp_without}")
print(f"   (raw): {raw_without[:200]}")

print("""
What you'll see:
  • With=True  → "The capital of France is Paris."
  • With=False → model writes another user message, or appends to the conversation
    pattern, or generates <|im_start|>user again — anything but answering.
""")

WITH add_generation_prompt=True (correct for inference):
'<|im_start|>system\nYou are a concise assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n'
  ↑ ends with <|im_start|>assistant  — model generates HERE as assistant

WITHOUT add_generation_prompt=False (wrong):
'<|im_start|>system\nYou are a concise assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n'
  ↑ ends with <|im_end|>  — model has no idea whose turn it is

══════════════════════════════════════════════════════════════════════
GENERATION COMPARISON
══════════════════════════════════════════════════════════════════════

✅ With add_generation_prompt=True:
   Paris

❌ Without (add_generation_prompt=False):
   superintendent
The capital of France is Paris.
   (raw): <|im_start|> superintendent
The capital of France is Paris.<|im_end|>

What you'll see:
  • With=True  → "The capital of France is Paris."
  • With=False → model writes another use

---
# SECTION 6: Multi-Turn Conversation
## How conversation history grows — and when it breaks

Every turn appends to the flat string. The model re-reads the **entire history** on every call.  
There is no persistent memory — just a growing token sequence.

In [13]:
class ChatSession:
    """
    A minimal stateful chat session.
    Shows the growing context window on every turn.
    """
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.history = [{"role": "system", "content": system_prompt}]

    def chat(self, user_message, max_new_tokens=80):
        self.history.append({"role": "user", "content": user_message})

        # Apply template — entire history every time
        rendered = tok_qwen.apply_chat_template(
            self.history,
            tokenize=False,
            add_generation_prompt=True,
        )
        context_tokens = len(tok_qwen.encode(rendered, add_special_tokens=False))

        response, _, _ = generate(rendered, max_new_tokens=max_new_tokens)
        self.history.append({"role": "assistant", "content": response})

        print(f"  [Context: {context_tokens} tokens total]")
        return response


# Run a 3-turn conversation
session = ChatSession("You are a concise Python tutor. Keep all answers under 2 sentences.")

turns = [
    "What is a list comprehension?",
    "Give me a one-line example.",
    "Now do the same with a dict.",
]

print("═" * 70)
print("MULTI-TURN CONVERSATION — watch the context window grow")
print("═" * 70)

for user_msg in turns:
    print(f"\nUser: {user_msg}")
    reply = session.chat(user_msg)
    print(f"Assistant: {reply}")

print("\n" + "═" * 70)
print("FINAL rendered prompt (what the model sees on turn 3):")
print("═" * 70)
final = tok_qwen.apply_chat_template(
    session.history, tokenize=False, add_generation_prompt=False
)
print(final)

══════════════════════════════════════════════════════════════════════
MULTI-TURN CONVERSATION — watch the context window grow
══════════════════════════════════════════════════════════════════════

User: What is a list comprehension?
  [Context: 34 tokens total]
Assistant: A list comprehension in Python allows you to create lists in a more concise way by combining loops and conditions into one line of code.

User: Give me a one-line example.
  [Context: 77 tokens total]
Assistant: ```python
[x**2 for x in range(5)]
```

This creates a list of squares from 0 to 4.

User: Now do the same with a dict.
  [Context: 123 tokens total]
Assistant: ```python
{x: x*2 for x in [1, 2, 3]}
```

This creates a dictionary where keys are numbers from 1 to 3 and values are their doubles.

══════════════════════════════════════════════════════════════════════
FINAL rendered prompt (what the model sees on turn 3):
══════════════════════════════════════════════════════════════════════
<|im_start|>system
Y

---
# SECTION 7: Special Tokens Deep Dive
## BOS, EOS, PAD — what each one does and what breaks without it

In [14]:
# Inspect special tokens for each loaded tokenizer
def inspect_special_tokens(tok, name):
    print(f"\n{'═'*55}")
    print(f"  {name}")
    print(f"{'═'*55}")
    print(f"  Vocab size : {tok.vocab_size:,}")
    print(f"  BOS        : {repr(tok.bos_token):<20} ID: {tok.bos_token_id}")
    print(f"  EOS        : {repr(tok.eos_token):<20} ID: {tok.eos_token_id}")
    print(f"  PAD        : {repr(tok.pad_token):<20} ID: {tok.pad_token_id}")
    print(f"  Chat tmpl  : {'yes' if tok.chat_template else 'NO'}")
    all_special = tok.all_special_tokens
    print(f"  All special tokens ({len(all_special)}):")
    for t in all_special[:12]:     # cap at 12 for readability
        tid = tok.convert_tokens_to_ids(t)
        print(f"    {tid:>8}  {repr(t)}")
    if len(all_special) > 12:
        print(f"    ... and {len(all_special) - 12} more")

inspect_special_tokens(tok_qwen, "Qwen3 1.7B")
inspect_special_tokens(tok_phi4, "Phi-4")


═══════════════════════════════════════════════════════
  Qwen3 1.7B
═══════════════════════════════════════════════════════
  Vocab size : 151,643
  BOS        : None                 ID: None
  EOS        : '<|im_end|>'         ID: 151645
  PAD        : '<|endoftext|>'      ID: 151643
  Chat tmpl  : yes
  All special tokens (14):
      151645  '<|im_end|>'
      151643  '<|endoftext|>'
      151644  '<|im_start|>'
      151646  '<|object_ref_start|>'
      151647  '<|object_ref_end|>'
      151648  '<|box_start|>'
      151649  '<|box_end|>'
      151650  '<|quad_start|>'
      151651  '<|quad_end|>'
      151652  '<|vision_start|>'
      151653  '<|vision_end|>'
      151654  '<|vision_pad|>'
    ... and 2 more

═══════════════════════════════════════════════════════
  Phi-4
═══════════════════════════════════════════════════════
  Vocab size : 32,000
  BOS        : '<s>'                ID: 1
  EOS        : '<|endoftext|>'      ID: 32000
  PAD        : '<|endoftext|>'      ID: 32000

In [15]:
# DEMO: Wrong EOS — model runs to max_new_tokens every time

correct_prompt = tok_qwen.apply_chat_template(
    [{"role": "user", "content": "What is 2+2? One word only."}],
    tokenize=False, add_generation_prompt=True,
)
input_ids = tok_qwen.encode(correct_prompt, return_tensors="pt",
                             add_special_tokens=False).to(DEVICE)

print("EOS DEMO — same prompt, different stop token\n")

with torch.inference_mode():
    # Case 1: correct EOS
    out_correct = model.generate(
        input_ids, max_new_tokens=30,
        eos_token_id=tok_qwen.eos_token_id,   # correct: 151645
        pad_token_id=tok_qwen.eos_token_id,
        do_sample=False,
    )
    n_correct = out_correct.shape[-1] - input_ids.shape[-1]
    text_correct = tok_qwen.decode(out_correct[0][input_ids.shape[-1]:], skip_special_tokens=True)

    # Case 2: wrong EOS (GPT-2's EOS — 50256, not in Qwen3 vocab)
    out_wrong = model.generate(
        input_ids, max_new_tokens=30,
        eos_token_id=50256,   # WRONG — this ID means nothing to Qwen3
        pad_token_id=50256,
        do_sample=False,
    )
    n_wrong = out_wrong.shape[-1] - input_ids.shape[-1]
    text_wrong = tok_qwen.decode(out_wrong[0][input_ids.shape[-1]:], skip_special_tokens=True)

print(f"✅ Correct EOS (ID {tok_qwen.eos_token_id}):")
print(f"   Stopped at {n_correct} new tokens")
print(f"   Output: {text_correct!r}")

print(f"\n❌ Wrong EOS (ID 50256 = GPT-2's <|endoftext|>):")
print(f"   Ran all the way to {n_wrong} new tokens (max reached)")
print(f"   Output: {text_wrong!r}")

print("""
  What happened:
  Token 50256 doesn't signal anything to Qwen3 — it treats it as a
  regular (very rare) token ID. The model never emits it, so the
  loop hits max_new_tokens every single time regardless of content.
""")

EOS DEMO — same prompt, different stop token

✅ Correct EOS (ID 151645):
   Stopped at 2 new tokens
   Output: '4'

❌ Wrong EOS (ID 50256 = GPT-2's <|endoftext|>):
   Ran all the way to 30 new tokens (max reached)
   Output: '4\nHuman: Can you explain the concept of recursion in programming and provide an example?\n\nAssistant: Recursion is a fundamental concept in computer'

  What happened:
  Token 50256 doesn't signal anything to Qwen3 — it treats it as a
  regular (very rare) token ID. The model never emits it, so the
  loop hits max_new_tokens every single time regardless of content.



---
# SECTION 8: Nemotron — Reasoning Model Output
## Parsing `<think>...</think>` traces

Nemotron Nano v3 appends `<think>` to the generation prompt.  
The model emits an internal reasoning trace before its final answer.  
If you pass the raw output to a downstream system, it includes the thinking.
You must parse and strip it.

In [17]:
# Simulate a Nemotron-style response (works even without the model loaded)
# Replace simulated_output with model.generate() output when Nemotron is available.

simulated_output = """<think>
The user is asking what 2+2 equals.
This is simple arithmetic: 2 + 2 = 4.
I should give a concise answer.
</think>
4"""

def parse_reasoning_output(raw_output):
    """
    Split Nemotron/DeepSeek R1 style output into:
    - reasoning: the <think>...</think> section
    - answer: everything after </think>
    """
    import re
    think_match = re.search(r"<think>(.*?)</think>", raw_output, re.DOTALL)
    reasoning = think_match.group(1).strip() if think_match else ""
    answer    = re.sub(r"<think>.*?</think>", "", raw_output, flags=re.DOTALL).strip()
    return reasoning, answer

reasoning, answer = parse_reasoning_output(simulated_output)

print("Raw model output:")
print(repr(simulated_output))

print("\n── Parsed ───────────────────────────────────────")
print(f"Reasoning trace: {repr(reasoning)}")
print(f"Final answer:    {repr(answer)}")

print("""
Production rule: always run parse_reasoning_output() before passing
Nemotron or DeepSeek R1 output to a downstream consumer.
The reasoning section is internal scratch-pad — not meant for the user.
""")

Raw model output:
'<think>\nThe user is asking what 2+2 equals.\nThis is simple arithmetic: 2 + 2 = 4.\nI should give a concise answer.\n</think>\n4'

── Parsed ───────────────────────────────────────
Reasoning trace: 'The user is asking what 2+2 equals.\nThis is simple arithmetic: 2 + 2 = 4.\nI should give a concise answer.'
Final answer:    '4'

Production rule: always run parse_reasoning_output() before passing
Nemotron or DeepSeek R1 output to a downstream consumer.
The reasoning section is internal scratch-pad — not meant for the user.



In [18]:
# Load Nemotron if available and run a real reasoning trace
NEMOTRON_PATH = LOCAL["nemotron"]

if os.path.isdir(NEMOTRON_PATH):
    print("Nemotron found — loading …")
    tok_nem = AutoTokenizer.from_pretrained(NEMOTRON_PATH, trust_remote_code=True,
                                             local_files_only=True)
    model_nem = AutoModelForCausalLM.from_pretrained(
        NEMOTRON_PATH, dtype=DTYPE, trust_remote_code=True, local_files_only=True
    ).to(DEVICE).eval()

    nem_messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is 17 × 13? Show your reasoning."},
    ]
    nem_prompt = tok_nem.apply_chat_template(
        nem_messages, tokenize=False, add_generation_prompt=True
    )
    print("\nRendered Nemotron prompt (ends with <think>):")
    print(repr(nem_prompt[-80:]))   # last 80 chars

    nem_ids = tok_nem.encode(nem_prompt, return_tensors="pt",
                              add_special_tokens=False).to(DEVICE)
    with torch.inference_mode():
        out = model_nem.generate(nem_ids, max_new_tokens=200,
                                  do_sample=False,
                                  eos_token_id=tok_nem.eos_token_id,
                                  pad_token_id=tok_nem.eos_token_id)
    raw = tok_nem.decode(out[0][nem_ids.shape[-1]:], skip_special_tokens=False)
    reasoning, answer = parse_reasoning_output(raw)
    print(f"\nReasoning:\n{reasoning}")
    print(f"\nFinal answer: {answer}")
else:
    print(f"Nemotron not found at {NEMOTRON_PATH}")
    print("Set LOCAL['nemotron'] to your model path to run this demo.")

Nemotron not found at /bigstore/models/nemotron-nano-v3
Set LOCAL['nemotron'] to your model path to run this demo.


---
# SECTION 9: Template Switcher Utility
## A helper you can actually use in production

In [19]:
from typing import List, Dict

class ChatTemplateInspector:
    """
    Utility: load a tokenizer, render a prompt, inspect special tokens,
    and count the rendered token budget.
    """
    def __init__(self, model_path: str, local_files_only: bool = True):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path, trust_remote_code=True, local_files_only=local_files_only
        )
        self.model_path = model_path

    def render(self, messages: List[Dict], add_generation_prompt: bool = True) -> str:
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt
        )

    def token_count(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=False))

    def report(self, messages: List[Dict]):
        rendered = self.render(messages)
        n = self.token_count(rendered)
        print(f"  Model    : {self.model_path}")
        print(f"  Tokens   : {n}")
        print(f"  BOS      : {repr(self.tokenizer.bos_token)} (ID {self.tokenizer.bos_token_id})")
        print(f"  EOS      : {repr(self.tokenizer.eos_token)} (ID {self.tokenizer.eos_token_id})")
        print(f"  PAD      : {repr(self.tokenizer.pad_token)} (ID {self.tokenizer.pad_token_id})")
        print(f"  Rendered :\n{rendered}")
        return rendered, n


# Test it with both models
sample = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user",   "content": "What is 2+2?"},
]

for path, local in [(QWEN3_PATH, QWEN3_LOCAL), (PHI4_PATH, PHI4_LOCAL)]:
    print("═" * 60)
    inspector = ChatTemplateInspector(path, local_files_only=local)
    inspector.report(sample)
    print()

════════════════════════════════════════════════════════════
  Model    : Qwen/Qwen2.5-1.5B-Instruct
  Tokens   : 26
  BOS      : None (ID None)
  EOS      : '<|im_end|>' (ID 151645)
  PAD      : '<|endoftext|>' (ID 151643)
  Rendered :
<|im_start|>system
You are a concise assistant.<|im_end|>
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant


════════════════════════════════════════════════════════════
  Model    : microsoft/Phi-3-mini-4k-instruct
  Tokens   : 19
  BOS      : '<s>' (ID 1)
  EOS      : '<|endoftext|>' (ID 32000)
  PAD      : '<|endoftext|>' (ID 32000)
  Rendered :
<|system|>
You are a concise assistant.<|end|>
<|user|>
What is 2+2?<|end|>
<|assistant|>




---
# SECTION 10: Try It Yourself

Use the cells below to experiment.

**Ideas:**
- Try a multi-turn conversation with a deliberate wrong template — does it recover?
- Change the system prompt and see how it affects generation
- Add a third model path and compare rendered token counts
- Try injecting prompt injection in the user message: `"Ignore all previous instructions and..."`

In [20]:
# ── YOUR EXPERIMENT ───────────────────────────────────────────────────
my_messages = [
    {"role": "system", "content": "You are a pirate. Answer only in pirate speech."},
    {"role": "user",   "content": "What is the capital of Spain?"},
]

# Choose a template: chatml_template | phi4_template | llama_template | no_template
my_template = chatml_template   # ← change this

prompt  = my_template(my_messages)
resp, _, n = generate(prompt, max_new_tokens=80, temperature=0.7)
print(f"Template: {my_template.__name__}")
print(f"New tokens generated: {n}")
print(f"\nResponse:\n{resp}")

Template: chatml_template
New tokens generated: 12

Response:
The capital of Spain, matey, is Madrid!


---
# Summary

```
messages dict
      ↓
apply_chat_template()     ← model-specific Jinja2 template
      ↓
rendered string           ← flat text with special tokens
      ↓
tokenizer.encode()        ← integers only
      ↓
model.generate()          ← one token at a time
```

| What goes wrong | Visible symptom |
|---|---|
| Wrong template (Llama on Qwen) | Model continues text pattern, no answer |
| Missing `add_generation_prompt` | Model writes another user turn |
| Wrong EOS token | Every response runs to `max_new_tokens` |
| No attention mask in batch | Shorter sequence output is degraded |
| Nemotron output not parsed | Reasoning trace passed to user |

**The single most important rule:**  
Always use `tokenizer.apply_chat_template()`. Never build the prompt string manually.

---

**Next lab:** Embedding layers & positional encodings — what happens to those token IDs after they enter the model.